# Candidate Generation: Content + Popularity Two-Tower MLP

A personalised candidate generator. Each request's causal history is mean-pooled
over valid text and entity vectors and encoded by a user tower. Each article is
encoded by a separate item tower from content plus frozen fit-split popularity
features. The logit is a normalised dot product plus a learned item bias, and a
sigmoid turns it into a ranking score.

Protocol:

- Use `full_history_rows` from the request profile for that impression. Never
  reuse a later request's profile for an earlier request.
- Exclude every article in the request's full causal history.
- Eligibility is `available_from <= timestamp`.
- Train on the sampled training rows, tune chronologically on the tail of train,
  then refit on all training rows and evaluate complete validation slates.
- Popularity features are frozen on the fit split, matching the popularity
  baseline. They are not per-request causal counts.
- The sigmoid is a ranking score. Training uses sampled exposed negatives, so it
  is not a calibrated click probability.
- Toggle `FULL_RUN` in the setup cell below: `True` runs the full training and
  evaluation; `False` runs the smoke pipeline for verification only.

## 1. Setup

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))
sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation" / "neural_models"))
sys.path.insert(0, str(ROOT / "src"))

from cg_evaluation import CandidateEvaluator
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
FULL_RUN = True
SMOKE = not FULL_RUN
RUN_TAG = "two_tower" if FULL_RUN else "two_tower_smoke"
MODEL_NAME = "content_popularity_two_tower_mlp"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
RECOMMENDATIONS_PATH = (
    ARTIFACT_DIR / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
)
RESULT_PATH = RESULTS / f"candidate_generation_{MODEL_NAME}_{RUN_TAG}_{TARGET_SPLIT}.csv"

SMOKE_TRAIN_ROWS = 300_000
SMOKE_TUNE_IMPRESSIONS = 1_000
SMOKE_REQUESTS = 3_000
MAX_EPOCHS = 12 if FULL_RUN else 2
PATIENCE = 3 if FULL_RUN else 1
HIDDEN_DIM = 256
EMBED_DIM = 128
DROPOUT = 0.1
BATCH_SIZE = 16_384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SEED = 4222
LOSS = "bce"  # Toggle here: "bpr" or "bce".
if LOSS not in {"bpr", "bce"}:
    raise ValueError(f"Unknown loss: {LOSS}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print("Mode:", "full" if FULL_RUN else "smoke")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

In [ ]:
TEXT_DIM = 64
ENTITY_DIM = 100
CONTENT_DIM = TEXT_DIM + ENTITY_DIM + 2
POPULARITY_DIM = 4
INPUT_DIM = CONTENT_DIM
ITEM_INPUT_DIM = CONTENT_DIM + POPULARITY_DIM

## 2. Data ingestion and model-related processing

### History pooling and feature builders

In [ ]:
def pool_profiles(history_rows, matrix, block=2000):
    """Mean-pool valid article vectors per request and L2-normalise."""
    matrix = np.asarray(matrix)
    if matrix.ndim != 2:
        raise ValueError("matrix must be two-dimensional")
    n = len(history_rows)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        total = int(lengths.sum())
        if total == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        if flat.min() < 0 or flat.max() >= matrix.shape[0]:
            raise ValueError("history contains an invalid article row")
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        valid_vectors = vectors[valid]
        starts = np.r_[
            0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1
        ]
        sums = np.add.reduceat(valid_vectors, starts, axis=0)
        counts = np.diff(np.r_[starts, len(valid_groups)])
        pooled = sums / counts[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result

In [ ]:
def profile_features(history_rows, text, entity, block=2000):
    """Build the pooled user input from causal history row references."""
    n = len(history_rows)
    out = np.zeros((n, INPUT_DIM), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        text_profile = pool_profiles(chunk, text, block=len(chunk))
        entity_profile = pool_profiles(chunk, entity, block=len(chunk))
        stop = start + len(chunk)
        out[start:stop, :TEXT_DIM] = text_profile
        out[start:stop, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity_profile
        out[start:stop, -2] = np.linalg.norm(text_profile, axis=1) > 0
        out[start:stop, -1] = np.linalg.norm(entity_profile, axis=1) > 0
    return out

In [ ]:
def item_features(text, entity, popularity=None):
    """Build article content inputs plus a frozen popularity block."""
    text = np.asarray(text, dtype=np.float32)
    entity = np.asarray(entity, dtype=np.float32)
    if text.shape[0] != entity.shape[0]:
        raise ValueError("text and entity arrays must align")
    n = text.shape[0]
    if popularity is None:
        popularity = np.zeros((n, POPULARITY_DIM), dtype=np.float32)
    else:
        popularity = np.asarray(popularity, dtype=np.float32)
        if popularity.shape != (n, POPULARITY_DIM):
            raise ValueError("popularity block shape does not align")

    out = np.zeros((n, ITEM_INPUT_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity
    out[:, CONTENT_DIM - 2] = np.linalg.norm(text, axis=1) > 0
    out[:, CONTENT_DIM - 1] = np.linalg.norm(entity, axis=1) > 0
    out[:, CONTENT_DIM:] = popularity
    return out

### Causal request profiles and pooled user inputs

The profile row for each impression contains the articles clicked before that
request. The cache below stores one 166-dimensional input per request:
`text64 profile`, `entity100 profile`, and two availability flags. Repeated
history references are preserved, zero vectors are ignored, and each block is
L2-normalised. The profile index also provides the `impression_uid -> row`
mapping used to join training rows.

In [ ]:
def load_profiles(split, validate=False):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No request profiles for {split}")
    frame = pl.read_parquet(
        parts,
        columns=["impression_uid", "user_code", "timestamp", "full_history_rows"],
    )
    if frame["impression_uid"].n_unique() != frame.height:
        raise ValueError("Duplicate impression_uid in request profiles")
    if frame["full_history_rows"].null_count():
        raise ValueError("Null full_history_rows in request profiles")
    frame = frame.sort("timestamp", maintain_order=True)
    if validate:
        impressions = pl.read_parquet(
            PROCESSED / f"impressions_{split}.parquet",
            columns=["impression_uid", "timestamp"],
        ).with_columns(pl.col("timestamp").cast(pl.Datetime("ns")))
        check = frame.select(
            "impression_uid",
            pl.col("timestamp").cast(pl.Datetime("ns")).alias("profile_timestamp"),
        ).join(impressions, on="impression_uid", how="full", coalesce=True)
        if check.height != frame.height:
            raise ValueError("Profile/impression join changed the row count")
        if check["profile_timestamp"].null_count() or check["timestamp"].null_count():
            raise ValueError("Missing profile or impression timestamp")
        if not (check["profile_timestamp"] == check["timestamp"]).all():
            raise ValueError("Profile timestamps do not match impressions")
    return frame

In [ ]:
def user_cache_paths(split):
    return (
        ARTIFACT_DIR / f"user_features_{split}.npy",
        ARTIFACT_DIR / f"profile_uids_{split}.npy",
        ARTIFACT_DIR / f"profile_index_{split}.parquet",
    )

In [ ]:
def load_user_cache(split, validate=False):
    feature_path, uid_path, index_path = user_cache_paths(split)
    if feature_path.exists() and uid_path.exists() and index_path.exists():
        features = np.load(feature_path, mmap_mode="r")
        uids = np.load(uid_path)
        if features.shape[0] != len(uids):
            raise ValueError("User feature cache is inconsistent")
        return features, uids

    frame = load_profiles(split, validate=validate)
    n = frame.height
    features = np.lib.format.open_memmap(
        feature_path, mode="w+", dtype=np.float32, shape=(n, INPUT_DIM)
    )
    for start in range(0, n, 20_000):
        end = min(start + 20_000, n)
        rows = frame["full_history_rows"].slice(start, end - start).to_list()
        features[start:end] = profile_features(rows, TEXT, ENTITY)
    features.flush()
    uids = frame["impression_uid"].to_numpy()
    np.save(uid_path, uids)
    pl.DataFrame(
        {
            "impression_uid": uids,
            "profile_row": np.arange(n, dtype=np.int32),
        }
    ).write_parquet(index_path)
    print(f"Built user cache for {split}: {n:,} requests")
    return np.load(feature_path, mmap_mode="r"), uids

In [ ]:
TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")

ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

train_profiles = load_profiles(FIT_SPLIT, validate=False)

train_user_features, train_profile_uids = load_user_cache(FIT_SPLIT)

profile_index_train = pl.read_parquet(user_cache_paths(FIT_SPLIT)[2])

if len(train_profile_uids) != train_profiles.height:
    raise ValueError("Train profile cache does not align with profiles")

if not np.array_equal(train_profile_uids, train_profiles["impression_uid"].to_numpy()):
    raise ValueError("Train profile cache UIDs do not match profiles")

print(f"Train requests: {train_profiles.height:,}")

print(f"Train user feature shape: {train_user_features.shape}")

del train_profiles

### Article content and popularity inputs, and the time-eligible catalogue

Article vectors are indexed by `article_row`, with row 0 reserved as padding.
The item tower receives content vectors, missing-vector flags, and a frozen
popularity block: log prior exposures, log prior clicks, smoothed CTR and a
has-prior-exposure flag. The popularity block is built from the fit split, so it
matches the popularity baseline's information set. The catalogue is ordered by
`available_from` so each request timestamp selects a prefix. Prior exposures are
also used for empty-history and short-list fallbacks.

In [ ]:
index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["article_row", "news_code"]
)
n_articles = index.height + 1
if TEXT.shape[0] != n_articles or ENTITY.shape[0] != n_articles:
    raise ValueError("Article vectors do not align with the catalogue")

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
catalogue = index.join(availability, on="news_code", how="left")
if catalogue["available_from"].null_count():
    raise ValueError("Missing available_from for some articles")

counts = pl.read_parquet(
    PROCESSED / "news_catalogue.parquet",
    columns=[
        "news_code",
        "exposures_train",
        "clicks_train",
        "exposures_validation",
        "clicks_validation",
    ],
)
prior_exposures = pl.col("exposures_train")
prior_clicks = pl.col("clicks_train")
if TARGET_SPLIT != "validation":
    prior_exposures = prior_exposures + pl.col("exposures_validation")
    prior_clicks = prior_clicks + pl.col("clicks_validation")

popularity = counts.select(
    "news_code",
    prior_exposures.alias("popularity_score"),
    prior_clicks.alias("prior_clicks"),
)
catalogue = catalogue.join(popularity, on="news_code", how="left")
if catalogue["popularity_score"].null_count():
    raise ValueError("Missing popularity_score for some articles")
if catalogue["prior_clicks"].null_count():
    raise ValueError("Missing prior_clicks for some articles")

ordered = catalogue.sort(["available_from", "news_code"])
article_rows = ordered["article_row"].to_numpy().astype(np.int64)
news_sorted = ordered["news_code"].to_numpy().astype(np.int64)
available_sorted = ordered["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = ordered["popularity_score"].to_numpy().astype(np.int64)

pop_rows = ordered["article_row"].to_numpy().astype(np.int64)
pop_exposures = ordered["popularity_score"].to_numpy().astype(np.float32)
pop_clicks = ordered["prior_clicks"].to_numpy().astype(np.float32)
pop_ctr = np.where(
    pop_exposures > 0,
    (pop_clicks + 1.0) / (pop_exposures + 20.0),
    0.0,
).astype(np.float32)
pop_block = np.zeros((n_articles, POPULARITY_DIM), dtype=np.float32)
pop_block[pop_rows, 0] = np.log1p(pop_exposures)
pop_block[pop_rows, 1] = np.log1p(pop_clicks)
pop_block[pop_rows, 2] = pop_ctr
pop_block[pop_rows, 3] = (pop_exposures > 0).astype(np.float32)

item_all = item_features(TEXT, ENTITY, pop_block)
item_sorted = item_all[article_rows]
pos_in_sorted = np.full(n_articles, -1, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(len(article_rows), dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

if not np.all(available_sorted[1:] >= available_sorted[:-1]):
    raise ValueError("Availability order is not non-decreasing")

print(f"Catalogue articles: {index.height:,}")
print(f"Item input shape: {item_all.shape}")
print(f"Popularity scores: {len(popularity_sorted):,}")
print(f"Articles with prior exposure: {int((pop_exposures > 0).sum()):,}")

### Training rows and chronological tuning slice

Training rows are the sampled exposed pairs. Labels are joined on both
`impression_uid` and `news_code`. The last 10% of training timestamps are held
out for early stopping; the tuning slates come from the complete
`candidates_train.parquet`, not from the sampled rows.

In [ ]:
feature_parts = sorted((FEATURES / f"ranking_features_{FIT_SPLIT}").glob("*.parquet"))
label_parts = sorted((FEATURES / f"labels_{FIT_SPLIT}").glob("*.parquet"))
if len(feature_parts) != len(label_parts):
    raise ValueError("Feature and label partitions do not align")

features = pl.read_parquet(
    feature_parts,
    columns=["impression_uid", "news_code", "article_row", "timestamp"],
)
labels = pl.read_parquet(
    label_parts, columns=["impression_uid", "news_code", "clicked"]
)
train = features.join(
    labels, on=["impression_uid", "news_code"], how="inner"
)
if train.height != features.height:
    raise ValueError("Feature/label join changed the row count")
train = train.join(profile_index_train, on="impression_uid", how="left")
if train["profile_row"].null_count():
    raise ValueError("Missing profile row for some training pairs")

ts_ns = train["timestamp"].to_numpy().astype("datetime64[ns]").astype(np.int64)
cutoff_ns = chronological_cutoff(ts_ns)
cutoff = np.datetime64(cutoff_ns, "ns")

all_user_idx = train["profile_row"].to_numpy().astype(np.int32)
all_item_idx = train["article_row"].to_numpy().astype(np.int32)
all_y = train["clicked"].to_numpy().astype(np.float32)
fit_mask = chronological_mask(ts_ns, cutoff_ns)

rng = np.random.default_rng(SEED)
fit_idx = np.flatnonzero(fit_mask)
if SMOKE and fit_idx.size > SMOKE_TRAIN_ROWS:
    fit_idx = rng.choice(fit_idx, SMOKE_TRAIN_ROWS, replace=False)
refit_idx = np.arange(len(all_y))
if SMOKE and refit_idx.size > SMOKE_TRAIN_ROWS:
    refit_idx = rng.choice(refit_idx, SMOKE_TRAIN_ROWS, replace=False)

if all_user_idx.max() >= train_user_features.shape[0]:
    raise ValueError("Training profile row outside the user cache")
if all_item_idx.max() >= item_all.shape[0]:
    raise ValueError("Training article row outside the item table")

print(f"Training pairs: {len(all_y):,}")
print(f"Chronological fit pairs: {len(fit_idx):,}")
print(f"Refit pairs: {len(refit_idx):,}")
print("Tuning cutoff:", cutoff)

TUNE_NDCG_K = max_clicks_per_impression(
    PROCESSED / "candidates_train.parquet"
)
tune = tuning_slates(
    PROCESSED / "candidates_train.parquet",
    cutoff_ns,
    index,
    profile_index_train,
)
if tune["article_row"].null_count() or tune["profile_row"].null_count():
    raise ValueError("Tuning rows have a missing article or profile")
if SMOKE:
    tune_uids_keep = (
        tune["impression_uid"].unique().shuffle(SEED + 1).head(SMOKE_TUNE_IMPRESSIONS)
    )
    tune = tune.filter(pl.col("impression_uid").is_in(tune_uids_keep.implode()))

tune_user_idx = tune["profile_row"].to_numpy().astype(np.int32)
tune_item_idx = tune["article_row"].to_numpy().astype(np.int32)
tune_y = tune["clicked"].to_numpy().astype(np.float32)
tune_uids_np = tune["impression_uid"].to_numpy()
print(f"Tuning impressions: {tune['impression_uid'].n_unique():,}")
print(f"Tuning pairs: {tune.height:,}")
print(f"Tuning NDCG k (max training clicks per impression): {TUNE_NDCG_K}")

## 3. Setting up the model

### Two-tower network definition

In [ ]:
class TwoTower(nn.Module):
    """Two independent MLP towers with a dot-product score and item bias."""

    def __init__(
        self,
        user_input_dim=INPUT_DIM,
        item_input_dim=ITEM_INPUT_DIM,
        hidden_dim=256,
        embed_dim=128,
        dropout=0.1,
        scale=10.0,
        max_scale=100.0,
        num_items=None,
    ):
        super().__init__()
        self.user_tower = self._tower(
            user_input_dim, hidden_dim, embed_dim, dropout
        )
        self.item_tower = self._tower(
            item_input_dim, hidden_dim, embed_dim, dropout
        )
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(scale)), dtype=torch.float32)
        )
        self.max_log_scale = float(np.log(max_scale))
        self.item_bias = (
            nn.Parameter(torch.zeros(num_items)) if num_items is not None else None
        )

    @staticmethod
    def _tower(input_dim, hidden_dim, embed_dim, dropout):
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def encode_users(self, values):
        return F.normalize(self.user_tower(values), dim=-1)

    def encode_items(self, values):
        return F.normalize(self.item_tower(values), dim=-1)

    def forward(self, user_values, item_values, item_indices=None):
        scale = torch.exp(torch.clamp(self.logit_scale, max=self.max_log_scale))
        logits = scale * (
            self.encode_users(user_values) * self.encode_items(item_values)
        ).sum(dim=-1)
        if self.item_bias is not None and item_indices is not None:
            logits = logits + self.item_bias[item_indices]
        return logits

## 4. Training the model

### Two-tower training, chronological early stopping and refit

The user tower consumes the 166-dimensional pooled history input. The item tower
consumes 170 dimensions: content, flags and the popularity block. The towers do
not share weights; their outputs are L2-normalised and combined by a scaled dot
product. A learned per-article bias captures popularity not explained by the
tower dot product. Early stopping uses mean slate NDCG@k on the chronological
tuning slice, with k set to the maximum number of clicks per impression in the
training slates; the final model is then refit on all training rows for the selected
epoch count. The objective is selected by `LOSS` in the setup cell: `"bce"` uses
binary cross-entropy on the sampled exposed pairs, while `"bpr"` pairs each click
with a non-click from the same request.

In [ ]:
def tune_metrics(model):
    model.eval()
    scores = np.empty(len(tune_y), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(tune_y), 100_000):
            end = min(start + 100_000, len(tune_y))
            users = torch.from_numpy(
                np.asarray(train_user_features[tune_user_idx[start:end]], dtype=np.float32)
            ).to(DEVICE)
            items = torch.from_numpy(item_all[tune_item_idx[start:end]]).to(DEVICE)
            indices = torch.from_numpy(
                tune_item_idx[start:end].astype(np.int64)
            ).to(DEVICE)
            scores[start:end] = model(users, items, indices).cpu().numpy()
    return mean_slate_metrics(
        tune_uids_np, tune_y, scores, TUNE_NDCG_K
    )

In [ ]:
def _bpr_pairs(user_idx, targets, rng):
    positive_rows = np.flatnonzero(targets > 0.5)
    negative_rows = np.flatnonzero(targets <= 0.5)
    order = np.argsort(user_idx[negative_rows], kind="stable")
    negative_rows = negative_rows[order]
    negative_users = user_idx[negative_rows]
    left = np.searchsorted(negative_users, user_idx[positive_rows], side="left")
    right = np.searchsorted(negative_users, user_idx[positive_rows], side="right")
    lengths = right - left
    keep = lengths > 0
    positive_rows = positive_rows[keep]
    left = left[keep]
    lengths = lengths[keep]
    if len(positive_rows) == 0:
        raise ValueError("No positive-negative pairs with the same user")
    picks = left + (rng.random(len(positive_rows)) * lengths).astype(np.int64)
    return positive_rows, negative_rows[picks]


def train_epoch(model, optimizer, user_idx, item_idx, targets, rng):
    model.train()
    if LOSS == "bpr":
        positive_rows, negative_rows = _bpr_pairs(user_idx, targets, rng)
        order = rng.permutation(len(positive_rows))
    else:
        order = rng.permutation(len(targets))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        if LOSS == "bpr":
            pos_rows = positive_rows[batch]
            neg_rows = negative_rows[batch]
            users = torch.from_numpy(
                np.asarray(
                    train_user_features[user_idx[pos_rows]], dtype=np.float32
                )
            ).to(DEVICE)
            positives = torch.from_numpy(item_all[item_idx[pos_rows]]).to(DEVICE)
            positive_indices = torch.from_numpy(
                item_idx[pos_rows].astype(np.int64)
            ).to(DEVICE)
            negatives = torch.from_numpy(item_all[item_idx[neg_rows]]).to(DEVICE)
            negative_indices = torch.from_numpy(
                item_idx[neg_rows].astype(np.int64)
            ).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            positive_scores = model(users, positives, positive_indices)
            negative_scores = model(users, negatives, negative_indices)
            loss = -F.logsigmoid(positive_scores - negative_scores).mean()
        else:
            users = torch.from_numpy(
                np.asarray(train_user_features[user_idx[batch]], dtype=np.float32)
            ).to(DEVICE)
            items = torch.from_numpy(item_all[item_idx[batch]]).to(DEVICE)
            indices = torch.from_numpy(item_idx[batch].astype(np.int64)).to(DEVICE)
            values = torch.from_numpy(targets[batch]).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(users, items, indices)
            loss = F.binary_cross_entropy_with_logits(logits, values)
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [ ]:
def new_model():
    return TwoTower(
        user_input_dim=INPUT_DIM,
        item_input_dim=ITEM_INPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        dropout=DROPOUT,
        num_items=n_articles,
    ).to(DEVICE)

In [ ]:
fit_user_idx = all_user_idx[fit_idx]

fit_item_idx = all_item_idx[fit_idx]

fit_y = all_y[fit_idx]

model = new_model()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED)

training_log = []

stopper = EarlyStopper(patience=PATIENCE)

for epoch in range(1, MAX_EPOCHS + 1):
    started = time.time()
    loss = train_epoch(model, optimizer, fit_user_idx, fit_item_idx, fit_y, rng)
    tune_auc, tune_ndcg = tune_metrics(model)
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "tune",
            "epoch": epoch,
            "train_loss": loss,
            "tune_auc": tune_auc,
            "tune_ndcg": tune_ndcg,
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(
        f"epoch {epoch:02d} loss={loss:.5f} "
        f"tune_auc={tune_auc:.5f} "
        f"tune_ndcg@{TUNE_NDCG_K}={tune_ndcg:.5f} "
        f"({elapsed:.1f}s)"
    )
    if stopper.step(tune_ndcg, model, epoch):
        print("Early stopping")
        break

stopper.restore(model)

best_epoch = stopper.best_epoch

print(
    f"Best epoch: {best_epoch} "
    f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
)

refit_user_idx = all_user_idx[refit_idx]

refit_item_idx = all_item_idx[refit_idx]

refit_y = all_y[refit_idx]

final_model = new_model()

optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED + 1)

for epoch in range(1, best_epoch + 1):
    started = time.time()
    loss = train_epoch(
        final_model, optimizer, refit_user_idx, refit_item_idx, refit_y, rng
    )
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "refit",
            "epoch": epoch,
            "train_loss": loss,
            "tune_auc": float("nan"),
            "tune_ndcg": float("nan"),
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(f"refit epoch {epoch:02d} loss={loss:.5f} ({elapsed:.1f}s)")

torch.save(
    {
        "state_dict": final_model.state_dict(),
        "config": {
            "user_input_dim": INPUT_DIM,
            "item_input_dim": ITEM_INPUT_DIM,
            "hidden_dim": HIDDEN_DIM,
            "embed_dim": EMBED_DIM,
            "dropout": DROPOUT,
            "num_items": n_articles,
            "popularity_dim": POPULARITY_DIM,
            "seed": SEED,
            "objective": LOSS,
            "best_epoch": best_epoch,
            "tune_ndcg": stopper.best_score,
            "ndcg_k": TUNE_NDCG_K,
        },
    },
    ARTIFACT_DIR / "model.pt",
)

pl.DataFrame(training_log).write_csv(ARTIFACT_DIR / "training_log.csv")

print("Saved model:", ARTIFACT_DIR / "model.pt")

## 5. Results

### Score the eligible catalogue and write Top-500 candidates

The item tower is evaluated once for the availability-sorted catalogue, and the
learned item bias is aligned to the same order. Each timestamp batch computes a
user-by-eligible-article score matrix plus the item bias. Articles in the
request's full causal history are excluded; empty-profile requests use the
popularity order. Ties are broken by popularity, then `news_code`.

In [ ]:
def popularity_select(order, n_eligible, excluded=None, k=500):
    """Take popular eligible positions from a precomputed global order."""
    if k < 1:
        raise ValueError("k must be positive")
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible:
            continue
        if position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f"only {len(selected)} popular eligible articles; cannot select {k}"
    )

In [ ]:
def _popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f"only {pool.size} eligible articles remain; cannot fill {count}"
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]

In [ ]:
def select_top_k(scores, popularity, news_code, k, excluded=None, fallback_order=None):
    """Deterministic Top-k by score, popularity, then news_code."""
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError("scores must be one-dimensional")
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError("score, popularity and news_code shapes must match")
    if k < 1:
        raise ValueError("k must be positive")
    n = scores.size
    if n < k:
        raise ValueError(f"only {n} eligible articles; cannot select {k}")
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError("excluded position outside the eligible array")
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return _popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort((news_code[ties], -popularity[ties]))
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = _popularity_fill(remaining, popularity, news_code, k - selected.size)
        selected = np.r_[selected, fill]

    return selected

In [ ]:
def encode_user_batches(model, features, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            end = min(start + batch, len(features))
            values = torch.from_numpy(
                np.array(features[start:end], dtype=np.float32, copy=True)
            ).to(DEVICE)
            out[start:end] = model.encode_users(values).cpu().numpy()
    return out

In [ ]:
def encode_item_batches(model, features, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            end = min(start + batch, len(features))
            values = torch.from_numpy(
                np.array(features[start:end], dtype=np.float32, copy=True)
            ).to(DEVICE)
            out[start:end] = model.encode_items(values).cpu().numpy()
    return out

In [ ]:
target_profiles = load_profiles(TARGET_SPLIT, validate=True)
target_user_features, target_profile_uids = load_user_cache(TARGET_SPLIT)
if not np.array_equal(
    target_profile_uids, target_profiles["impression_uid"].to_numpy()
):
    raise ValueError("Target profile cache does not align with profiles")

model = final_model
model.eval()
user_emb = encode_user_batches(model, target_user_features)
item_emb = encode_item_batches(model, item_sorted)
item_bias_sorted = None
if model.item_bias is not None:
    item_bias_sorted = model.item_bias.detach().cpu().numpy()[article_rows]

has_profile = (
    (target_user_features[:, -2] > 0) | (target_user_features[:, -1] > 0)
)

impression_uids = target_profiles["impression_uid"].to_numpy()
timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
history_rows = target_profiles["full_history_rows"].to_list()
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
pairs = list(zip(starts, ends))
if SMOKE:
    batch_rng = np.random.default_rng(SEED)
    batch_order = batch_rng.permutation(len(pairs))
    selected = []
    total = 0
    for index in batch_order:
        start, end = pairs[index]
        selected.append((start, end))
        total += end - start
        if total >= SMOKE_REQUESTS:
            break
    pairs = sorted(selected)
processed_requests = int(sum(end - start for start, end in pairs))
print(f"Scoring requests: {processed_requests:,} / {len(timestamps):,}")

cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(RECOMMENDATIONS_PATH)
processed_uids = []
started = time.time()
try:
    for batch_no, (start, end) in enumerate(pairs, 1):
        timestamp = timestamps[start]
        n_eligible = int(np.searchsorted(available_sorted, timestamp, side="right"))
        if n_eligible < CANDIDATE_CAP:
            raise ValueError(
                f"Only {n_eligible} eligible articles at {timestamp}"
            )
        scores = user_emb[start:end] @ item_emb[:n_eligible].T
        if item_bias_sorted is not None:
            scores = scores + item_bias_sorted[:n_eligible][None, :]
        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = news_sorted[:n_eligible]
        selected_batch = np.empty(
            (end - start, CANDIDATE_CAP), dtype=np.int64
        )
        for local, profile_row in enumerate(range(start, end)):
            rows = history_rows[profile_row]
            if rows:
                excluded = pos_in_sorted[np.asarray(rows, dtype=np.int64)]
                excluded = excluded[
                    (excluded >= 0) & (excluded < n_eligible)
                ]
            else:
                excluded = np.empty(0, dtype=np.int64)
            fallback = (
                None if has_profile[profile_row] else global_popularity_order
            )
            selected_batch[local] = select_top_k(
                scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=fallback,
            )

        uids = impression_uids[start:end]
        writer.add(
            np.repeat(uids, CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            news_batch[selected_batch.reshape(-1)],
        )
        processed_uids.append(uids.copy())
        if batch_no % 1000 == 0 or batch_no == len(pairs):
            print(
                f"Scored {batch_no:,}/{len(pairs):,} timestamp batches; "
                f"{writer.rows:,} rows",
                flush=True,
            )
finally:
    writer.close()

elapsed = time.time() - started
expected_rows = processed_requests * CANDIDATE_CAP
actual_rows = pl.scan_parquet(RECOMMENDATIONS_PATH).select(pl.len()).collect().item()
if actual_rows != expected_rows:
    raise ValueError(
        f"Expected {expected_rows:,} rows, found {actual_rows:,}"
    )
if SMOKE:
    check = pl.read_parquet(RECOMMENDATIONS_PATH)
    if check.select(pl.col("cutoff").n_unique()).item() != CANDIDATE_CAP:
        raise ValueError("Smoke output is missing cutoff values")
    if check.group_by("impression_uid").agg(
        pl.col("news_code").n_unique().alias("unique_news")
    ).filter(pl.col("unique_news") != CANDIDATE_CAP).height:
        raise ValueError("Smoke output contains duplicate candidates")
subset_uids = (
    np.concatenate(processed_uids)
    if processed_uids
    else np.empty(0, dtype=np.int64)
)
print(f"Wrote {actual_rows:,} candidate rows in {elapsed:.1f}s")
print("Candidates:", RECOMMENDATIONS_PATH)

### Manifest, evaluation and comparison

Full runs use the shared `CandidateEvaluator`, which validates impression
coverage, cutoffs, uniqueness and time eligibility. Smoke runs score only a
request subset, so they compute subset recall with the same metric columns and
write `smoke_recall_<split>.csv` instead of calling the shared evaluator.

In [ ]:
def smoke_recall(recommendations_path, model_name, impression_uids):
    uid_frame = pl.DataFrame(
        {"impression_uid": np.asarray(impression_uids, dtype=np.int64)}
    )
    label_parts = sorted(
        (FEATURES / f"labels_{TARGET_SPLIT}").glob("*.parquet")
    )
    if not label_parts:
        raise FileNotFoundError(f"No labels for {TARGET_SPLIT}")
    labels = pl.read_parquet(
        label_parts,
        columns=["impression_uid", "news_code", "clicked"],
    ).join(uid_frame, on="impression_uid", how="inner")
    positives = labels.filter(pl.col("clicked") == 1).select(
        "impression_uid", "news_code"
    )
    relevant = positives.group_by("impression_uid").agg(
        pl.len().alias("relevant")
    )
    base = uid_frame.join(
        relevant, on="impression_uid", how="left"
    ).with_columns(pl.col("relevant").fill_null(0).cast(pl.Int64))
    candidates = pl.read_parquet(
        recommendations_path,
        columns=["impression_uid", "news_code", "cutoff"],
    )
    rows = []
    for m in M_VALUES:
        selected = candidates.filter(pl.col("cutoff") <= m)
        hits = (
            positives.join(
                selected.select("impression_uid", "news_code"),
                on=["impression_uid", "news_code"],
                how="inner",
            )
            .group_by("impression_uid")
            .agg(pl.len().alias("hits"))
        )
        per_impression = base.join(
            hits, on="impression_uid", how="left"
        ).with_columns(pl.col("hits").fill_null(0).cast(pl.Int64))
        relevant_clicks = int(per_impression["relevant"].sum())
        retrieved_relevant = int(per_impression["hits"].sum())
        with_relevant = per_impression.filter(pl.col("relevant") > 0)
        recall_impression_mean = (
            float(
                (
                    with_relevant["hits"] / with_relevant["relevant"]
                ).mean()
            )
            if with_relevant.height
            else float("nan")
        )
        hit_rate = (
            float((per_impression["hits"] > 0).sum() / per_impression.height)
            if per_impression.height
            else float("nan")
        )
        rows.append(
            {
                "model": model_name,
                "split": TARGET_SPLIT,
                "m": int(m),
                "recall_pooled": (
                    retrieved_relevant / relevant_clicks
                    if relevant_clicks
                    else float("nan")
                ),
                "recall_impression_mean": recall_impression_mean,
                "hit_rate": hit_rate,
                "impressions": int(per_impression.height),
                "relevant_clicks": relevant_clicks,
                "retrieved_relevant": retrieved_relevant,
                "mean_candidates": float(min(m, CANDIDATE_CAP)),
            }
        )
    return pl.DataFrame(rows)

In [ ]:
smoke_recall_path = None
if SMOKE:
    smoke_evaluation = smoke_recall(
        RECOMMENDATIONS_PATH, MODEL_NAME, subset_uids
    )
    smoke_recall_path = ARTIFACT_DIR / f"smoke_recall_{TARGET_SPLIT}.csv"
    smoke_evaluation.write_csv(smoke_recall_path)
    print(smoke_evaluation.sort("m"))
    print("Smoke recall:", smoke_recall_path)

In [ ]:
manifest = {
    "model": MODEL_NAME,
    "score": "sigmoid(scaled content dot product + learned item bias; item input includes fit-split popularity)",
    "popularity_features": [
        "log1p(prior_exposures)",
        "log1p(prior_clicks)",
        "smoothed_ctr=(clicks+1)/(exposures+20)",
        "has_prior_exposure",
    ],
    "popularity_source": f"{FIT_SPLIT} catalogue aggregates (plus validation for test targets)",
    "profile_source": f"4_Feature_Engineering/artifacts/prequential-all/request_profiles_{TARGET_SPLIT}",
    "profile_column": "full_history_rows",
    "profile_pooling": "valid-vector mean per text/entity block, L2 normalised; no decay",
    "history_policy": "exclude every article in full_history_rows",
    "fallback": "popularity order (prior exposures) for empty profiles or short lists",
    "fit_split": FIT_SPLIT,
    "target_split": TARGET_SPLIT,
    "candidate_cap": CANDIDATE_CAP,
    "candidate_format": ["impression_uid", "cutoff", "news_code"],
    "eligibility": "available_from <= timestamp",
    "mode": "full" if FULL_RUN else "smoke",
    "seed": SEED,
    "device": DEVICE,
    "torch_version": torch.__version__,
    "user_input_dim": INPUT_DIM,
    "item_input_dim": ITEM_INPUT_DIM,
    "train_pairs": int(len(all_y)),
    "fit_pairs": int(len(fit_idx)),
    "refit_pairs": int(len(refit_idx)),
    "best_epoch": int(best_epoch),
    "tune_ndcg": float(stopper.best_score),
    "ndcg_k": int(TUNE_NDCG_K),
    "target_requests": int(processed_requests),
    "recommendation_rows": int(actual_rows),
    "recommendations_file": RECOMMENDATIONS_PATH.name,
    "smoke_recall_file": (
        smoke_recall_path.name if smoke_recall_path is not None else None
    ),
    "model_file": "model.pt",
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(ARTIFACT_DIR / f"manifest_{TARGET_SPLIT}.json").write_text(
    json.dumps(manifest, indent=2) + "\n"
)
print("Manifest:", ARTIFACT_DIR / f"manifest_{TARGET_SPLIT}.json")

In [ ]:
if FULL_RUN:
    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate(
        RECOMMENDATIONS_PATH,
        model_name=MODEL_NAME,
        m_values=M_VALUES,
    )
    evaluation.write_csv(RESULT_PATH)
    comparison_path = (
        RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"
    )
    if comparison_path.exists():
        existing = pl.read_csv(comparison_path).filter(
            pl.col("model") != MODEL_NAME
        )
        evaluation = evaluation.select(existing.columns)
        comparison = pl.concat([existing, evaluation])
    else:
        comparison = evaluation
    comparison.write_csv(comparison_path)
    print(evaluation.sort("m"))
    print("Results:", RESULT_PATH)
else:
    print(
        "Smoke run: shared evaluator skipped; subset recall written."
    )

## Notes and limitations

- The user representation is specific to each request. Repeat users do not
  share a single latest profile; earlier requests use only earlier feedback.
- This is a content + popularity model, not matrix factorisation: there are no
  user-ID or article-ID embeddings.
- Excluding the causal history removes repeat-click opportunities, matching the
  content baseline. Recall therefore does not measure repeat clicks.
- The popularity block and item bias use fit-split aggregates. For validation
  they are frozen at the end of train; they are not per-request causal counts.
- The sigmoid is a ranking score. Sampled exposed negatives make the training
  positive rate much higher than held-out CTR, so the output is not calibrated.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.